# Greenhouse Gas Emissions Explorer

### 1. Introduction
This dashboard explores trends in greenhouse gas (GHG) pollution by the world's 10 largest emitters. The dashboard visualizes countries' climate pollution over time and provides a detailed breakdown of the sectors, sources, and gases responsible for those emissions. You can find the finished dashboard [here](https://ghg-explorer-svpx.onrender.com) and the GitHub repo for this project [here](https://github.com/sallcj/ghg-emissions-explorer). A video walkthrough of the dashboard is [here](https://drive.google.com/file/d/1SJ-qL7bD_WcyK7chmAH0QQTNFNkhKEm7/view?usp=sharing).

![Screenshot of dashboard](data/screenshot.png)

### 2. Library choice

The dashboard was created in Python using [Panel](https://panel.holoviz.org/) and the [HoloViews](https://holoviews.org/index.html) plotting library. Panel is an open-source Python library for building interactive data dashboards that handles app layout and provides controls for the plots created with other libraries. I've chosen Panel for this project because it integrates seamlessly with Jupyter notebooks, so that I can illustrate the step-by-step process of building the dashboard and serve up the final product from this notebook, without additional export steps. HoloViews is a high-level wrapper built on top of other plotting libraries such as [Bokeh](https://bokeh.org/), [Plotly](https://plotly.com/python/), and [Matplotlib](https://matplotlib.org/stable/). It follows a 'data-first' approach to visualization that its developers claim is more intuitive and concise. It's also built and maintained by the same core team that created Panel and shares many underlying libraries, so Panel and HoloViews are a natural fit. I've built other dashboards using Plotly + Dash in the past, so I've chosen HoloViews and its Bokeh for this project to experiment with its features.

Let's start by downloading our libraries and setting our configurations.

In [1]:
# Libraries and initializations
import numpy as np
import pandas as pd
import panel as pn
from urllib.request import urlretrieve
import holoviews as hv
import hvplot.pandas 
import matplotlib as mpl
from matplotlib.colors import to_hex, to_rgb
import warnings
import world_bank_data as wb
from pathlib import Path
from bokeh.core.properties import value
from bokeh.models import (ColumnDataSource, GlyphRenderer, GraphRenderer, HoverTool, Label,
                          ColorBar, FixedTicker, LabelSet, Line, NumeralTickFormatter, Patches, Text)
from cmcrameri import cm

warnings.simplefilter('ignore')
hv.extension("bokeh")
pn.extension()

# Data directory
DATA_DIR = Path("data")

### 3. Data
For the dashboard, I'll use the September 2026 release of the [PRIMAP-hist](https://zenodo.org/records/22876287) dataset by by Johannes Gütschow and coauthors. The dataset is provided by Gütschow and Climate Resource on Zenodo under a non-commercial (CC BY-NC-SA) license and contains detailed estimates of greenhouse gas (GHG) emissions for more than 200 countries and regions from 1750 to 2025. 

I'll start by downloading a CSV file with all the PRIMAP-hist data and reading it into a Pandas DataFrame. We'll need the annual totals for emissions by source sector and pollutant for each country. The main source sectors, as defined by [Intergovernmental Panel for Climate Change (IPCC)](https://www.ipcc.ch/), include:
- energy use (burning fossil fuels),
- industrial processes,
- agriculture,
- waste, and 
- a catch-all 'other' category.

I won't include GHG emissions and removals from lands and forests.

The pollutants are all the primary GHGs covered by the [Kyoto Protocol](https://unfccc.int/process-and-meetings/the-kyoto-protocol) and include:
- carbon dioxide (CO<sub>2</sub>),
- methane (CH<sub>4</sub>),
- nitrous oxide (N<sub>2</sub>O),
- hydrofluorocarbons (HFCs),
- perfluorocarbons (PFCs), and
- sulphur hexafluoride (SF<sub>6</sub>). 

HFCs, PFCs, and SF<sub>6</sub> are grouped together as F-gases.  Total emissions for each gas are weighted by their [global warming potential](https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Glossary:Global-warming_potential_(GWP)) (i.e., their relative potency in trapping heat) and reported in equivalent units of CO<sub>2</sub> (CO<sub>2</sub>e).

The columns/variables in the processed DataFrame that I'll use for the dashboard are:

| Column name | Variable Definition |
|-------------|---------------------|
| iso3 | ISO 3166-1 alpha-3 codes for countries and regions |
| name | Country or region name |
| sector | IPCC source sector (Energy, industrial processes, agriculture, waste, other, or total) |
| gas | Species of GHG emitted (CO<sub>2</sub>, CH<sub>4</sub>, N<sub>2</sub>O, or F-gases) |
| unit | Unit (Mt/kt CO2-equivalent) |
| year | Year (1990-2025) |
| emissions | Annual emissions |

In [2]:
# Define function for downloading, unzipping, and reading in EDGAR_2026_GHG data
def load_primap_data(url: str, filename: str, outfile,
                     country_list_filename, data_dir=DATA_DIR, force_download=False):
    """
    Download the PRIMAP-hist data, read it into Pandas, and clean it
    
    Parameters
    ------------
    url : str
        The URL containing the PRIMAP-hist data
    filename : str
        Name of the CSV file we want to download
    outfile : str
        Name of the processed CSV file (which should already be saved in the Git repo)
    data_dir : str, optional
        File path name for the raw data
    force_download : bool, optional
        If True, download the zip file even if it already exists

    Returns
    ------------
    A Pandas DataFrame with the annual emissions data for all countries, sectors, and gases we want
    """
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    
    # Check if the input and output CSV files exist, and download them if needed
    in_filepath, out_filepath = data_dir / filename, data_dir / outfile

    # Use the CSV file with the already processed data if it exists (to avoid lengthy downloads)...
    if out_filepath.exists():
            df = pd.read_csv(out_filepath, index_col=0)
            df["year"] = df["year"].astype(int)
            return df

    # ...or download and clean the CSV file
    else:
        if force_download or not in_filepath.exists():
            print(f"Downloading {filename}...")
            urlretrieve(url, in_filepath)
        else:
            print(f"Data file already exists. Using existing file for {filename}")

        # Read in the raw CSV file
        df = pd.read_csv(in_filepath)

        # Only use the "HISTCR" data - corresponds with country-reported data series
        df = df[df["scenario (PRIMAP-hist)"] == "HISTCR"]

        # Pollutants - only keep aggregate baskets/groups of Kyoto Protocol gases
        gases = ["CH4", "CO2", "KYOTOGHG (AR4GWP100)", "N2O", "FGASES (AR4GWP100)"]
        df = df[df['entity'].isin(gases)] 
        df['entity'] = df['entity'].replace({ # simplify the names of gases
            "CH4": "ch4",
            "CO2": "co2", 
            "KYOTOGHG (AR4GWP100)": "total",
            "N2O": "n2o",
            "FGASES (AR4GWP100)": "fgas"
        })
        
        # Ignore regional groupings (other than the EU)
        exclude_grps = ["ANNEXI", "AOSIS", "BASIC", "EARTH", "LDC", "NONANNEXI", "UMBRELLA"]
        df = df[~df["area (ISO3)"].isin(exclude_grps)]
        df["area (ISO3)"] = df["area (ISO3)"].replace("EU27BX", "EUU") # use the EU's ISO3 code

        # Only keep the columns we need
        df = df.drop(columns=["source", "scenario (PRIMAP-hist)", "provenance"])

        # Rename columns we keep
        df = df.rename(columns={
            "area (ISO3)": "iso3",
            "category (IPCC2006_PRIMAP)": "sector",
            "entity": "gas"
            })

        # Reshape to long format, using only the columns we need
        id_cols = ["iso3", "gas", "unit", "sector"]
        year_cols = [y for y in df.columns if y not in id_cols]
        df = df.melt(id_vars=id_cols, value_vars=year_cols, var_name="year", value_name="emissions")
        df["year"] = df["year"].astype(int)

        # The PRIMAP data go back to 1750 - let's just keep years since 1990
        df = df[df["year"] >= 1990].reset_index(drop=True)

        # Match ISO3 codes to country names
        country_list_filepath = data_dir / country_list_filename
        country_list_df = pd.read_csv(country_list_filepath)
        country_list_df.columns = country_list_df.columns.str.lower()
        df = df.merge(country_list_df[['iso3', 'name']], how="inner", on="iso3")

        df = df.reindex(
             columns=['iso3', 'name', 'sector', 'gas', 'unit', 'year', 'emissions']
        )
        
        # Export CSV file with the cleaned data
        df.to_csv(out_filepath)

        return df


# Load the data - as_cached keeps ONE copy for the whole server, instead of one per visitor.
primap_df = pn.state.as_cached(
    "primap_df", load_primap_data,
    url="https://zenodo.org/records/22876287/files/Guetschow_et_al_2026-PRIMAP-hist_v2.8_final_no_rounding_22-Sep-2026.csv?download=1",
    filename="Guetschow_et_al_2026-PRIMAP-hist_v2.8_final_no_rounding_22-Sep-2026.csv",
    outfile="primap_hist_cleaned.csv",
    country_list_filename="country_list.csv",
    force_download=False
)
primap_df.head()


,iso3,name,sector,gas,unit,year,emissions
0,ABW,Aruba,0,ch4,CH4 * gigagram / yr,1990,0.529719
1,ABW,Aruba,1,ch4,CH4 * gigagram / yr,1990,0.038726
2,ABW,Aruba,1.A,ch4,CH4 * gigagram / yr,1990,0.025406
3,ABW,Aruba,1.B,ch4,CH4 * gigagram / yr,1990,0.013320
4,ABW,Aruba,1.B.1,ch4,CH4 * gigagram / yr,1990,0.013201


The data include estimated emissions for each granular sub-sector. The `sector` codes correspond with the GHG source categories defined by the [Intergovernmental Panel on Climate Change (IPCC)](https://www.ipcc.ch/). The full list of sectors in the PRIMAP-hist data is provided [here](https://zenodo.org/records/22876287). Let's just get the annual totals and sector aggregates.

In [3]:
# Annual totals by country for all pollutants and sectors
tot_mask = (primap_df['sector'] == "M.0.EL") & (primap_df['gas'] == "total")
ghg_tot = primap_df[tot_mask].reset_index(drop=True)

# Annual totals by country for all pollutants and each sector
aggregated_sectors = ['1', '2', '4', '5', 'M.AG', 'M.0.EL']
ghg_sector_gas = primap_df[primap_df.sector.isin(aggregated_sectors)].reset_index(drop=True)

# For CO2 and N2O, replace physical units with CO2-equivalents
# Note: We'll apply the IPCC AR4 Global Warming Potential (GWP) values
ar4_gwp = {
    'ch4': 25, # each ton of CH4 has the effect of 28 tons of CO2
    'n2o': 298 
}
ghg_sector_gas['emissions'] = ghg_sector_gas['emissions'] * ghg_sector_gas['gas'].map(ar4_gwp).fillna(1) / 1000.0
ghg_sector_gas['unit'] = "Mt CO2e" # all gases now in units of Mt CO2e per year, since we're focused on big countries
ghg_sector_gas.head()

,iso3,name,sector,gas,unit,year,emissions
0,ABW,Aruba,1,ch4,Mt CO2e,1990,0.000968
1,ABW,Aruba,4,ch4,Mt CO2e,1990,0.010993
2,ABW,Aruba,M.0.EL,ch4,Mt CO2e,1990,0.013243
3,ABW,Aruba,M.AG,ch4,Mt CO2e,1990,0.001282
4,ABW,Aruba,1,co2,Mt CO2e,1990,0.491333


The PRIMAP-hist data cover over 200 countries and regions. Let's focus on the top 10 countries/regions by total emissions in 2025. The number-one country, China (CHN), emitted the equivalent of 15.6 billion tons CO<sub>2</sub> last year. That's about 2.5x the U.S. total. India (IND), the third largest polluter, the equivalent of around 3.9 billion tons CO<sub>2</sub>. Together, these 10 countries and regions alone account for nearly three-quarters of global GHG emissions.

In [4]:
# Identify the top 10 regions by their 2025 emissions
ghg_tot_copy = ghg_tot.copy()
latest_yr = ghg_tot_copy.year.max()
top10_df = (ghg_tot_copy[ghg_tot_copy.year == latest_yr]
    .sort_values("emissions", ascending=False)
    .head(10).reset_index(drop=True))

# Top 10 countries' share of global GHG emissions
top10_emissions = top10_df['emissions'].sum().astype(float)
global_emissions = ghg_tot_copy[(ghg_tot_copy['iso3'] != "EUU") & (ghg_tot_copy.year == latest_yr)]['emissions'].sum().astype(float)
top10_emissions_share = top10_emissions / global_emissions * 100.0
print(f"Top-10 countries' share of global GHG emissions in {latest_yr}: {top10_emissions_share:.1f}%")

top10_df


Top-10 countries' share of global GHG emissions in 2025: 72.1%


,iso3,name,sector,gas,unit,year,emissions
0,CHN,China,M.0.EL,total,CO2 * gigagram / yr,2025,15548824.0
1,USA,USA,M.0.EL,total,CO2 * gigagram / yr,2025,6309173.4
2,IND,India,M.0.EL,total,CO2 * gigagram / yr,2025,3862920.2
3,EUU,European Union,M.0.EL,total,CO2 * gigagram / yr,2025,2975462.0
4,RUS,Russia,M.0.EL,total,CO2 * gigagram / yr,2025,2116667.6
5,BRA,Brazil,M.0.EL,total,CO2 * gigagram / yr,2025,1255193.7
6,IDN,Indonesia,M.0.EL,total,CO2 * gigagram / yr,2025,1206707.4
7,IRN,Iran,M.0.EL,total,CO2 * gigagram / yr,2025,1195982.7
8,JPN,Japan,M.0.EL,total,CO2 * gigagram / yr,2025,1021937.6
9,SAU,Saudi Arabia,M.0.EL,total,CO2 * gigagram / yr,2025,804023.5


Let's see how the list of top 10 emitters has shifted over the years. Because I've included the EU as a region, I'll drop the individual EU27 member states.

In [5]:
# Get rankings by year, using country names, dropping individual EU member states
ghg_tot_copy = ghg_tot.copy()
country_list_filepath = DATA_DIR.joinpath("country_list.csv")
country_list_df = pd.read_csv(country_list_filepath)
country_list_df.columns = country_list_df.columns.str.lower()


years = [1990, 2000, 2000, 2010, 2020, 2025]
rankings_dict = {}
for yr in years:
    merged_df = ghg_tot_copy.merge(country_list_df[['iso3', 'eu27']], how="left", on="iso3")
    merged_df['name'] = np.where(
        merged_df['iso3'] == "EUU",
        "European Union",
        merged_df['name']
    )
    no_eu_members = merged_df[merged_df['eu27'] != 1]
    top10 = (no_eu_members[no_eu_members['year'] == yr]
    .sort_values("emissions", ascending=False)
    .head(10)
    .reset_index(drop=True))
    rankings_dict[str(yr)] = top10["name"].tolist()

rankings_df = pd.DataFrame(rankings_dict)
rankings_df.head(10)

,1990,2000,2010,2020,2025
0,USA,USA,China,China,China
1,European Union,China,USA,USA,USA
2,China,European Union,European Union,European Union,India
3,Russia,Russia,India,India,European Union
4,Japan,India,Russia,Russia,Russia
5,India,Japan,Japan,Brazil,Brazil
6,Ukraine,Brazil,Brazil,Japan,Indonesia
7,United Kingdom,Canada,Iran,Iran,Iran
8,Brazil,United Kingdom,Canada,Indonesia,Japan
9,Canada,Mexico,Mexico,Saudi Arabia,Saudi Arabia


How do emissions by the top-10 countries stack up on a per-capita basis? Let's bring in population data from the World Bank and compare. The World Bank data are from the [World Development Indicators](https://datacatalog.worldbank.org/search/dataset/0037712/world-development-indicators), fetched via the `wbgapi` package, and are provided under a [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

In [6]:
# Get population data via World Bank API 
def get_wb_pop_data(indicator="SP.POP.TOTL", out_file='wb_pop_data.csv',
                    data_dir=DATA_DIR, force_download=False):
        data_dir = Path(data_dir)
        data_dir.mkdir(parents=True, exist_ok=True)
        out_filepath = data_dir.joinpath(out_file)
        if force_download or not out_filepath.exists():
                df = (
                    wb.get_series(indicator, id_or_value="id", date="1990:2025")
                      .rename("pop")
                      .reset_index()
                )
                # Make column names consistent with the emissions data
                df = df.rename(columns={"Country": 'iso3', "Year": 'year'})
                df = df.drop(columns='Series')
                df["year"] = df["year"].astype(int)
                df.to_csv(out_filepath)
                return df
        else:
                df = pd.read_csv(out_filepath, index_col=0)
                return df

pop_df = get_wb_pop_data()

In [7]:
# Per-capita and total GHG emissions
df_copy = ghg_tot.copy()
ghg_tot_percap = df_copy.merge(pop_df, how="inner", on=["iso3", "year"])
ghg_tot_percap['emissions_percap'] = ghg_tot_percap['emissions'] / ghg_tot_percap['pop'] * 1000 # kilotons -> tons
ghg_tot_percap['emissions'] = ghg_tot_percap['emissions'] / 1000 # kilotons -> million tons
keepcols = ['iso3', 'name', 'year', 'emissions', 'emissions_percap']
ghg_tot_percap = ghg_tot_percap.loc[:, keepcols]
ghg_tot_percap.reindex(columns=keepcols)

# Names of the top-10 countries
top10_iso3 = top10_df['iso3'].unique().tolist()

# How the top-10 emitters from above stack up on per-capita basis
mask = (ghg_tot_percap['iso3'].isin(top10_iso3)) & (ghg_tot_percap['year'] == 2025)
top10_percap_df = ghg_tot_percap[mask].sort_values('emissions_percap', ascending=False).reset_index(drop=True)
top10_percap_df

,iso3,name,year,emissions,emissions_percap
0,SAU,Saudi Arabia,2025,804.0235,21.745907
1,USA,USA,2025,6309.1734,18.459488
2,RUS,Russia,2025,2116.6676,14.748927
3,IRN,Iran,2025,1195.9827,12.941059
4,CHN,China,2025,15548.8240,11.054308
5,JPN,Japan,2025,1021.9376,8.283737
6,EUU,European Union,2025,2975.4620,6.595613
7,BRA,Brazil,2025,1255.1937,5.898123
8,IDN,Indonesia,2025,1206.7074,4.223373
9,IND,India,2025,3862.9202,2.638849


The rankings look much different now comparing tons of GHGs emitted per capita. I'll highlight these differences with my visualizations, comparing how the biggest polluters' emissions vary on a total and per-capita basis. I also want to dig a bit deeper to understand how the composition and sources of pollution in these countries vary.

## 4. Visualization techniques

I'll make four types of charts using [HoloViews](https://holoviews.org/#) and [Bokeh](https://bokeh.org/):

- **Line plots**: I'll include line plots for both total and per-capita emissions to illustrate the rapid increase in China's emissions and how it rose from being the third to the first largest polluter by the 2010s. The line plots would look like a mess of spaghetti if I tried to include all 200+ countries and regions covered by the PRIMAP-hist dataset. I'll limit the chart to the top-10 countries and regions and use colored highlights to call out trends for a few key countries. I've chosen the United States, China, and Saudi Arabia to illustrate how Saudi Arabia and US emissions are so much higher than China's if you consider population.

- **Box charts**: The box charts are a complement to the line plots. They'll show emissions in the starting and ending years for all 10 countries included in the line plots to provide an easy to read, side-by-side measure of the magnitude by which the different countries' emissions have increased or fallen.

- **Sankey diagram**: Comparing total vs per-capita emissions gives a basic sense of how some economies and societies are much more pollution intensive than others. I'll go a step farther by adding a Sankey diagram next to dissect the contrasting composition of emissions in these 10 countries. Sankey diagrams are useful for decomposing processes and flows. They're often used to show how energy is transformed and consumed by different sectors in an economy. I'm building on that same idea here to show flows of pollution.

- **Heatmap**: Heatmaps use color as a preattentive attribute to provide a quick impression of contrasting patterns or where data are clustered. I'll make a heatmap to show how the shares of emissions of specific pollutants by different source sectors vary across the different countries. I want to show, for example, how a superpollutant like methane comes from much different sources in a country like Iran versus one like India or Brazil. The aim here is to continue to provide a more nuanced visual understanding of how the sources of pollution vary so much in these big economies, and where there are points of commonality. 

In the next section, I'll walk through the process of building each of these visualizations and combining them into a cohesive dashboard.

## 5. Building and exploring

### 5.1. Common settings and styling

I'll begin by defining some variables and basic chart settings that I'll use for several of the charts.

In [8]:
# Names of the top-10 countries
top10_name = top10_percap_df['name'].unique().tolist()

# Range of years for all plots
FIRST, LAST = int(ghg_tot_percap.year.min()), int(ghg_tot_percap.year.max())

# Default plot width and height
WIDTH, HEIGHT = 600, 350

# Variables used for both the line plots and bar charts
VARIABLES = {
    "emissions":  {"label": "Million tons CO₂e", "tick_fmt": "0,0", "tip_fmt": "0,0"},
    "emissions_percap": {"label": "Tons CO₂e per person", "tick_fmt": "0.[0]", "tip_fmt": "0.0"},
}

# Fonts dictionary for titles and labels used across all the dashboard components and charts
FONTS = {
    "title":            {"font": "helvetica", "size": "14px", "style": "normal", "color": "#374151"},  # chart titles
    "axis_labels":      {"font": "helvetica", "size": "12px", "style": "normal", "color": "#4b5563c0"},  # tick labels: numbers, years, countries, sectors
    "axis_title":       {"font": "helvetica", "size": "12px", "style": "normal", "color": "#4b5563c0"},  # axis titles (when a chart shows one)
    "line_labels":      {"font": "helvetica", "size": "12px", "style": "bold",   "color": "#374151"},  # names at the end of the lines (colour comes from HIGHLIGHT)
    "node_labels":      {"font": "helvetica", "size": "12px", "style": "normal", "color": "#111827"},  # Sankey node labels ("Energy  9,394")
    "column_headings":  {"font": "helvetica", "size": "14px", "style": "bold",   "color": "#374151"},  # Sankey "Total / Sector / Gas"
    "legend":           {"font": "helvetica", "size": "12px", "style": "normal", "color": "#374151"},  # bar-chart legend, heatmap colour scale
    "tooltip":          {"font": "helvetica", "size": "12px", "style": "normal", "color": "#111827"},  # hover callouts (a bold name line is kept)
}

For all the colors, I'll use Fabio Crameri's ['Scientific Colour Maps'](https://www.fabiocrameri.ch/colourmaps/), which are beautiful, [perceptually uniform](https://s-ink.org/perceptually-uniform-colour-gradients), colorblind-friendly color gradients. I'll define an overall color scheme and create a helper function to split up color maps into evenly spaced, discrete bands.

In [9]:
# Set overall colormap for line plots and bar charts
COLORS = cm.navia

# Utility function for picking colors
def get_colors(bands: int, cmap=cm.navia, lo=.2, hi=.8):
    '''
    Function to split color maps into evenly spaced bands for discrete/qualitative color mapping

    Parameters
    ------------
    bands : int
        Specified number of discrete color bands/categories
    cmap : Matplotlib colormap (optional)
        The chosen color map from the cmcrameri library

    Returns
    ------------
    colors : list
        A list of strings with HEX values for the colors for each
        
    Notes
    ------------
        Works best with a sequential color map.
        See list of color maps at https://www.fabiocrameri.ch/colourmaps-userguide/
    '''
    # Create equally spaced intervals between 20% and 80% of the colormap range (by default)
    intervals = np.linspace(lo, hi, bands)

    # Divide up the colormap into discrete bins
    colors = [mpl.colors.to_hex(cmap(val)) for val in intervals]

    return colors

# Define countries for highlighting in the line plots and assign them colors 
highlight_countries = ["USA", "China", "Saudi Arabia"]
highlight_colors = get_colors(len(highlight_countries), cmap=COLORS)
HIGHLIGHT = dict(zip(highlight_countries, highlight_colors))

# Assign colors for other countries in the line plots
GREY = "#babfc7"
HOVER_DARK = "#23262C"


Next are a set of functions that will strip down the Bokeh defaults and create a minimalist style, supressing toolbars and grids. The functions also define the styles for text callouts in tooltips upon hoving over the chart feastures. The charts will all call these style settings using hooks.

In [10]:
# Define minimalist styling
def apply_minimal_style(fig, variable):
    keep_only_hover(fig)
    fig.outline_line_color = None
    fig.xgrid.visible = False
    fig.ygrid.visible = False
    fig.xaxis.axis_line_color = "#9ca3af"
    fig.xaxis.major_tick_line_color = "#9ca3af"
    fig.yaxis.axis_line_color = None
    fig.yaxis.major_tick_line_color = None
    fig.axis.minor_tick_line_color = None      # no minor tick marks
    apply_font(fig.axis, "axis_labels", "major_label_text_")
    apply_font(fig.axis, "axis_title", "axis_label_text_")
    fig.yaxis.formatter = NumeralTickFormatter(format=VARIABLES[variable]["tick_fmt"])
    fig.min_border_right = 24

# Font styling and properties
def font_props(role, prefix="text_"):
    # FONTS[role] -> Bokeh text properties, e.g. {"text_font": ..., "text_font_size": ..., ...}.
    # `prefix` picks which Bokeh text it is for: "text_" (titles, labels), "major_label_text_" (axis ticks) ...
    f = FONTS[role]
    return {f"{prefix}font": f["font"], f"{prefix}font_size": f["size"],
            f"{prefix}font_style": f["style"], f"{prefix}color": f["color"]}


def apply_font(obj, role, prefix="text_"):
    # Set all four text properties of a Bokeh object (an axis, title, colour bar ...) from FONTS[role].
    for name, value in font_props(role, prefix).items():
        setattr(obj, name, value)


# Tooltip and callout textbox styling and functionality
def tooltip_box(html):
    # Wrap callout HTML in a box that uses the "tooltip" font settings.
    f = FONTS["tooltip"]
    return (f'<div style="font: {f["style"]} {f["size"]} {f["font"]}, sans-serif; color: {f["color"]}; '
            f'padding: 2px 4px;">{html}</div>')


def keep_only_hover(fig):
    # Keep ONLY the hover tool: no pan, wheel-zoom, box-zoom, tap, save or reset.
    fig.toolbar.tools = [t for t in fig.toolbar.tools if isinstance(t, HoverTool)]
    fig.toolbar.active_drag = None
    fig.toolbar.active_scroll = None
    fig.toolbar.active_tap = None
    fig.toolbar_location = None


def make_tooltip(variable):
    # HTML callout: country, then year and value
    fmt = VARIABLES[variable]["tip_fmt"]
    return tooltip_box(
        '<div style="font-weight: 600;">@name</div>'
        f'<div>@year{{0}}: <b>@{variable}{{{fmt}}}</b></div>'
    )

### 5.2. Line Plots

Now for the line plots of the total and per-capita emissions time series for the top-10 emitters, with colored highlights for USA, China, and Saudi Arabia. 

Let's define the functions for building the basic components of the line plots. The functions will create chart objects that plot the time series for each country and then overlay them. I'll also create a hook that will add highlight individual countries' lines upon hovering along with callouts for emissions in individual years.

In [11]:
# Make sure the year range is valid (this will come in handy once the slider is added)
def clean_range(year_range):
    # (first, last) years, at least one year apart and inside the data.
    first, last = sorted(int(y) for y in year_range)
    if first >= last:
        first, last = (first, first + 1) if first < LAST else (LAST - 1, LAST)
    return max(first, FIRST), min(last, LAST)

# Create hook for applying styles to line plot
def make_line_hook(variable, year_range=(FIRST, LAST)):
    first, last = clean_range(year_range)
    last_values = ghg_tot_percap[ghg_tot_percap.year == last].set_index("name")[variable]
    tooltip = make_tooltip(variable)

    def hook(plot, element):
        fig = plot.state
        if "styled" in fig.tags:          # hooks can run again on updates; only do this once
            return
        fig.tags = fig.tags + ["styled"]

        # Remove HoloViews' default hover tool
        for tool in [t for t in fig.toolbar.tools if isinstance(t, HoverTool)]:
            fig.toolbar.tools.remove(tool)

        # One hover glyph per line: what the line looks like while hovered
        lines = [r for r in fig.renderers
                 if isinstance(r, GlyphRenderer) and isinstance(r.glyph, Line)]
        for r in lines:
            g = r.glyph
            is_grey = str(g.line_color).lower() == GREY
            r.hover_glyph = Line(
                x=g.x, y=g.y,
                line_color=HOVER_DARK if is_grey else g.line_color,
                line_width=2.5 if is_grey else 3.5,
                line_alpha=1,
            )

        # Hover tool: only the nearest line, snapped to its nearest data point
        fig.add_tools(HoverTool(renderers=lines, tooltips=tooltip,
                                line_policy="nearest", mode="mouse"))

        # Direct labels at the right-hand end of the highlighted lines
        for name, color in HIGHLIGHT.items():
            fig.add_layout(Label(
                x=last + 0.011 * (last - first), y=float(last_values[name]), text=name,
                text_baseline="middle", **{**font_props("line_labels"), "text_color": color},
            ))

        apply_minimal_style(fig, variable)

    return hook

# Create chart objects for individual countries' time series
def country_curve(name, variable, year_range=(FIRST, LAST)):
    # One hvPlot line for one country, for the years in year_range.
    first, last = clean_range(year_range)
    sub = ghg_tot_percap[(ghg_tot_percap.name == name) & ghg_tot_percap.year.between(first, last)]
    highlighted = name in HIGHLIGHT
    return sub.hvplot.line(
        x="year", y=variable,
        hover_cols=["name"],               # puts the country name into the hover data
        label=name,
        color=HIGHLIGHT.get(name, GREY),
        line_width=2.5 if highlighted else 1.2,
        alpha=1.0 if highlighted else 0.9,
    )

# Overlay the line plots
def make_line_chart(variable, year_range=(FIRST, LAST)):
    # Line chart of `variable` for all countries, for the years in year_range.
    first, last = clean_range(year_range)
    span = last - first
    step = 1 if span <= 10 else 2 if span <= 20 else 5          # tick spacing that suits the span
    others = [c for c in top10_name if c not in HIGHLIGHT]
    draw_order = others + list(HIGHLIGHT)      # grey lines first so the coloured ones sit on top
    return hv.Overlay([country_curve(c, variable, (first, last)) for c in draw_order]).opts(
        width=WIDTH, height=HEIGHT,
        show_legend=False,
        xlim=(first, last + 0.27 * span),     # extra room on the right for the labels
        ylim=(0, None),
        xticks=list(range(first, last + 1, step)),
        xlabel="", ylabel=VARIABLES[variable]["label"],
        hooks=[make_line_hook(variable, (first, last))],
    )

Let's see a sample chart for per capita emissions from 1990 to 2025.

In [12]:
chart_percap = make_line_chart("emissions_percap", year_range=(1990, 2025))
chart_percap

:Overlay
   .Curve.Russia         :Curve   [year]   (emissions_percap,name)
   .Curve.Iran           :Curve   [year]   (emissions_percap,name)
   .Curve.Japan          :Curve   [year]   (emissions_percap,name)
   .Curve.European_Union :Curve   [year]   (emissions_percap,name)
   .Curve.Brazil         :Curve   [year]   (emissions_percap,name)
   .Curve.Indonesia      :Curve   [year]   (emissions_percap,name)
   .Curve.India          :Curve   [year]   (emissions_percap,name)
   .Curve.USA            :Curve   [year]   (emissions_percap,name)
   .Curve.China          :Curve   [year]   (emissions_percap,name)
   .Curve.Saudi_Arabia   :Curve   [year]   (emissions_percap,name)

### 5.3. Bar Charts

The bar charts will show emissions for each of the top-10 countries in the first and last years. I'll make a hook for the bar chart to apply stylings and then create the actual chart.

In [13]:
# Bar charts: the two years to compare (first and last years from the line plot)
BAR_YEARS = [FIRST, LAST]
BAR_COLORS = get_colors(2, cmap=COLORS, lo=.3, hi=.7) # same color scheme, draw from middle of gradient

def make_bar_hook(variable):
    tooltip = make_tooltip(variable)           # same callout style as the line charts

    def hook(plot, element):
        fig = plot.state
        apply_minimal_style(fig, variable)
        fig.xaxis.major_tick_line_color = None
        fig.xaxis.major_label_text_font_size = "0px"    # hide the repeated 1990 / 2025 labels (legend covers it)
        fig.xaxis.separator_line_color = None           # no lines between country groups
        apply_font(fig.xaxis, "axis_labels", "group_text_")     # country names
        fig.xaxis.group_label_orientation = 0.6                 # slanted (radians) so neighbours don't collide
        for legend in fig.legend:
            legend.border_line_color = None
            apply_font(legend, "legend", "label_text_")
        for tool in fig.toolbar.tools:
            if isinstance(tool, HoverTool):
                tool.tooltips = tooltip
    return hook


def make_bar_chart(variable, year_range=(FIRST, LAST)):
    # Grouped bar chart of `variable` for each country in the first and last year of year_range.
    copy_df = ghg_tot_percap.copy()
    years = list(clean_range(year_range))
    mask = copy_df.year.isin(years) & copy_df.name.isin(top10_name) # only the top-10 countries!
    sub = copy_df[mask]
    sub["year"] = sub["year"].astype(str)              # text, so bars group and colour by year
    year_labels = [str(y) for y in years]

    # Countries sorted by the later year's value (largest first)
    order = (sub[sub.year == year_labels[-1]]
             .sort_values(variable, ascending=False)["name"].tolist())

    # Declare the dimensions with their values to fix the order. (Not redim.values(name=...):
    # a dimension called "name" clashes with an argument inside HoloViews' redim.)
    kdims = [hv.Dimension("name", values=order), hv.Dimension("year", values=year_labels)]
    bars = hv.Bars(sub, kdims=kdims, vdims=[variable])
    return bars.opts(
        width=WIDTH, height=HEIGHT,
        color="year", cmap=BAR_COLORS,
        line_color=None, bar_width=0.9,
        tools=["hover"],                  # the hook then restyles its callout
        hover_line_color=HOVER_DARK, hover_line_width=1.5,
        legend_position="top_right",
        ylim=(0, None), xlabel="", ylabel=VARIABLES[variable]["label"],
        hooks=[make_bar_hook(variable)],
    )

As an example, let's see per capita emissions for 1990 and 2025.

In [14]:
bar_emissions_percap = make_bar_chart("emissions_percap", year_range=(1990, 2025))
bar_emissions_percap

:Bars   [name,year]   (emissions_percap)

### 5.4. Alluvial chart (AKA, a Sankey diagram)

The next set of charts will drill down on individual countries to show the sources and composition of their GHG emissions vary. I'll create an alluvial plot with three columns. The chart will flow from total emissions on the left to total emissions are divided by sector in the middle and then how each of those sectors emissions are broken up by gas on the right.

HoloViews doesn't have a separate `Alluvial` element like some other plotting libraries, so I'll build the chart using `Sankey`. First I'll set up the data and define the categories that will be included in each of the column's nodes and define the colors and column titles.

In [15]:
VALUE_COL = "emissions"       
TOTAL_SECTOR = "M.0.EL"       # sector code for the national total
TOTAL_GAS = "total"      
TOTAL_LABEL = "Total emissions"

# Optional display names; any code not listed is shown as it appears in the data
SECTOR_LABELS = {"1": "Energy", "2": "Industrial processes", "M.AG": "Agriculture",
                "4": "Waste", "5": "Other"}
GAS_LABELS = {"co2": "CO₂", "ch4": "CH₄", "n2o": "N₂O", "fgas": "F-gases", "total": "All gases"}

SANKEY_WIDTH, SANKEY_HEIGHT = 900, 520

# Headings under the three columns (left to right) and how they look
COLUMN_TITLES = ["Total", "Sector", "Gas"]
# (their font, size, style and colour are FONTS["column_headings"] in the settings cell)
COLUMN_TITLE_PADDING = 8            # extra space (pixels) reserved under the chart for the headings


def sector_label(code):
    return SECTOR_LABELS.get(code, code)


def gas_label(code):
    return GAS_LABELS.get(code, code)


# Headings under the three columns (left to right) and how they look
COLUMN_TITLES = ["Total", "Sector", "Gas"]
COLUMN_TITLE_STYLE = dict(
    text_font="helvetica",          # any font installed on the viewer's computer, e.g. "Georgia", "Courier New"
    text_font_size="14px",          # "12px", "16px", ...
    text_font_style="bold",         # "normal", "bold", "italic" or "bold italic"
    text_color="#374151",           # any colour name or hex code
    text_alpha=1.0,                 # 0 (invisible) to 1 (solid)
)
COLUMN_TITLE_PADDING = 0            # extra space (pixels) reserved under the chart for the headings

TOTAL_COLOR = "#8c919a"  # medium grey for the left node
SECTOR_CMAP = COLORS # use the line and box plot colors for the sector nodes
GAS_CMAP = cm.lajolla # add a secondary colormap for the gas nodeas


# Get a full list of nodes
sectors = sorted(s for s in ghg_sector_gas["sector"].unique() if s != TOTAL_SECTOR)
gases = sorted(g for g in ghg_sector_gas["gas"].unique() if g != TOTAL_GAS)
all_nodes = [TOTAL_LABEL] + [sector_label(s) for s in sectors] + [gas_label(g) for g in gases]
assert len(set(all_nodes)) == len(all_nodes), "a sector, a gas and the total share a display name"


# Zip together a dictionary of colors for all the nodes
sector_colors = get_colors(len(sectors), cmap=SECTOR_CMAP, lo=.25, hi=.75)
gas_colors = get_colors(len(gases), cmap=GAS_CMAP, lo=.25, hi=.75)
NODE_COLORS = dict(zip(all_nodes, [TOTAL_COLOR] + sector_colors + gas_colors))
NODE_COLORS

{'Total emissions': '#8c919a',
 'Energy': '#1b608f',
 'Industrial processes': '#2f798b',
 'Waste': '#418a80',
 'Other': '#5aa171',
 'Agriculture': '#89c36a',
 'CH₄': '#67342a',
 'CO₂': '#bb4b48',
 'F-gases': '#e27b50',
 'N₂O': '#edae54'}

The `sankey_links` function turns each country-year into a Sankey diagram table with `source → target, value` linking the three dimensions and nodes in the diagram.

In [16]:
def _country_year(name, year):
    return ghg_sector_gas[(ghg_sector_gas["name"] == name) & (ghg_sector_gas["year"] == year)]


def national_total(name, year):
    d = _country_year(name, year)
    row = d[(d["sector"] == TOTAL_SECTOR) & (d["gas"] == TOTAL_GAS)][VALUE_COL]
    if len(row) != 1:
        raise ValueError(f"expected one {TOTAL_SECTOR}/{TOTAL_GAS} row for {name}, {year}; found {len(row)}")
    return float(row.iloc[0])


def sankey_links(name, year):
    d = _country_year(name, year)
    if d.empty:
        raise ValueError(f"no data for {name}, {year}")
    by_sector = d[(d["gas"] == TOTAL_GAS) & (d["sector"] != TOTAL_SECTOR)]
    by_sector_gas = d[(d["gas"] != TOTAL_GAS) & (d["sector"] != TOTAL_SECTOR)]

    left = pd.DataFrame({                      # total -> sector
        "source": TOTAL_LABEL,
        "target": by_sector["sector"].map(sector_label),
        "value": by_sector[VALUE_COL],
    })
    right = pd.DataFrame({                     # sector -> gas
        "source": by_sector_gas["sector"].map(sector_label),
        "target": by_sector_gas["gas"].map(gas_label),
        "value": by_sector_gas[VALUE_COL],
    })
    links = pd.concat([left, right], ignore_index=True)
    return links[links["value"] > 0].reset_index(drop=True)   # a Sankey cannot draw zero/negative flows


Next, define how each node will be sliced into bands and how the colors will be blend for the links connecting nodes.

In [17]:
def band_slices(xs, ys, n_slices=60):
    # Cut one flow polygon into thin vertical slices. Returns a list of (xs, ys, t), where t runs
    # from 0 at the left end of the flow to 1 at the right end.
    xs, ys = np.asarray(xs, float), np.asarray(ys, float)
    left, right = xs.min(), xs.max()
    at_right = np.flatnonzero(xs == right)
    lower_x, lower_y = xs[1:at_right[0] + 1], ys[1:at_right[0] + 1]      # edge drawn left -> right
    upper_x, upper_y = xs[at_right[-1]:][::-1], ys[at_right[-1]:][::-1]  # edge drawn right -> left; flip it
    edges = np.linspace(left, right, n_slices + 1)
    low, up = np.interp(edges, lower_x, lower_y), np.interp(edges, upper_x, upper_y)
    return [([edges[i], edges[i + 1], edges[i + 1], edges[i]],
             [low[i], low[i + 1], up[i + 1], up[i]],
             (i + 0.5) / n_slices) for i in range(n_slices)]


def blend(colour_a, colour_b, t, strength=0.6):
    # Colour a fraction t of the way from a to b, softened towards white so labels stay readable.
    mix = (1 - t) * np.array(to_rgb(colour_a)) + t * np.array(to_rgb(colour_b))
    return to_hex(1 - strength * (1 - mix))

Then define the functions for piecing together the elements of the Sankey chart. The hook adds the color gradient, formats the labels, hides center circles at smaller nodes, and sets up tooltips.

In [18]:
def make_sankey_hook(node_totals, grand_total):
    box = tooltip_box("%s")
    edge_tip = box % ('<div style="font-weight: 600;">@source &rarr; @target</div>'
                      '<div><b>@value{0,0} Mt CO₂e</b> &nbsp;(@share{0.0%} of total)</div>')
    node_tip = box % ('<div style="font-weight: 600;">@name</div>'
                      '<div><b>@value{0,0} Mt CO₂e</b> &nbsp;(@share{0.0%} of total)</div>')

    def hook(plot, element):
        fig = plot.state
        if "styled" in fig.tags:
            return
        fig.tags = fig.tags + ["styled"]

        keep_only_hover(fig)
        fig.outline_line_color = None
        apply_font(fig.title, "title")

        graph = [r for r in fig.renderers if isinstance(r, GraphRenderer)][0]
        quad = [r for r in fig.renderers if r.__class__.__name__ == "GlyphRenderer"
                and type(r.glyph).__name__ == "Quad"][0]
        graph.node_renderer.visible = False          # hide the circles drawn at node centres

        # Labels: "Name  1,234" on a soft white background (replaces HoloViews' own labels)
        for r in [r for r in fig.renderers if isinstance(getattr(r, "glyph", None), Text)]:
            data, glyph = r.data_source.data, r.glyph
            keep = [i for i, t in enumerate(data["text"]) if t]
            fig.add_layout(LabelSet(
                x="x", y="y", text="text",
                source=ColumnDataSource({
                    "x": [data["x"][i] for i in keep],
                    "y": [data["y"][i] for i in keep],
                    "text": [f"{data['text'][i]}\n{node_totals[data['text'][i]]:,.0f} Mt CO₂e" for i in keep],
                }),
                text_align=glyph.text_align, text_baseline=glyph.text_baseline, text_line_height=1.0,
                background_fill_color="white", background_fill_alpha=0.8,
                **{**font_props("node_labels"),
                   "text_font": value(FONTS["node_labels"]["font"])},   # value(): a font NAME, not a data column
            ))
            r.visible = False

        # Column headings under the three node columns (min_border_bottom leaves room for them)
        heading_px = float(FONTS["column_headings"]["size"].replace("px", ""))
        fig.min_border_bottom = int(COLUMN_TITLE_PADDING + 28 + heading_px)     # grows with the font size
        centres = sorted({round((a + b) / 2, 3) for a, b in zip(quad.data_source.data["x0"],
                                                                 quad.data_source.data["x1"])})
        for centre, title in zip(centres, COLUMN_TITLES):
            fig.add_layout(Label(x=centre, y=COLUMN_TITLE_PADDING - 2, x_units="data", y_units="screen",
                                 text=title, text_align="center", text_baseline="bottom",
                                 **font_props("column_headings")))

        # Extra columns used by the callouts
        names = [str(n) for n in element.nodes.dimension_values("index")]   # same order as the node rows
        quad.data_source.data = {
            **quad.data_source.data, "name": names,
            "value": [node_totals[n] for n in names],
            "share": [node_totals[n] / grand_total for n in names],
        }
        edge_renderer = graph.edge_renderer
        edges = edge_renderer.data_source.data
        edge_renderer.data_source.data = {
            **edges, "share": [v / grand_total for v in edges["value"]]}

        # Gradient: one patch per slice, drawn underneath the (now invisible) flow shapes
        patch_x, patch_y, patch_colour = [], [], []
        for k in range(len(edges["value"])):
            colour_a, colour_b = NODE_COLORS[edges["source"][k]], NODE_COLORS[edges["target"][k]]
            for sx, sy, t in band_slices(edges["xs"][k], edges["ys"][k]):
                patch_x.append(sx)
                patch_y.append(sy)
                patch_colour.append(blend(colour_a, colour_b, t))
        gradient = GlyphRenderer(
            data_source=ColumnDataSource({"xs": patch_x, "ys": patch_y, "colour": patch_colour}),
            glyph=Patches(xs="xs", ys="ys", fill_color="colour", line_color="colour", line_width=0.7))
        fig.renderers = [gradient] + list(fig.renderers)

        # The real flow shapes stay (so hovering works) but are see-through; hovering outlines the flow
        edge_renderer.glyph = Patches(xs="xs", ys="ys", fill_alpha=0, line_alpha=0)
        edge_renderer.hover_glyph = Patches(xs="xs", ys="ys", fill_alpha=0,
                                            line_color="#111827", line_width=1.5)
        edge_renderer.selection_glyph = edge_renderer.nonselection_glyph = None

        for tool in fig.toolbar.tools:               # the existing hover tool covers the flows
            tool.tooltips = edge_tip
            tool.renderers = [edge_renderer]
        fig.add_tools(HoverTool(renderers=[quad], tooltips=node_tip))   # and one for the nodes

    return hook


def make_sankey(name, year):
    # Sankey of total -> sector -> gas for one country and year.
    links = sankey_links(name, year)            # raises a clear error if there is no data
    total = national_total(name, year)
    node_totals = {TOTAL_LABEL: total, **links.groupby("target")["value"].sum().to_dict()}
    return hv.Sankey(links, kdims=["source", "target"], vdims=["value"]).opts(
        width=SANKEY_WIDTH, height=SANKEY_HEIGHT,
        node_color="index", cmap=NODE_COLORS,
        label_position="outer", show_values=False,
        node_width=30, node_padding=36,
        node_line_width=0, edge_line_width=0,
        hooks=[make_sankey_hook(node_totals, total)],
    )

Last, create a sample chart for China's emissions in 2025.

In [19]:
sankey_china_2025 = make_sankey("China", 2025)
sankey_china_2025

:Sankey   [source,target]   (value)

The Sankey chart is by far the most complicated of the charts and the one that required leaning on AI the most to build. It also had the biggest influence on my choice of data. I originally planned to use a data series with both GHG emissions and removals (i.e., one that included the CO<sub>2</sub> absorbed by forests and lands, but this made it impossible to show a Sankey chart for countries such as China with net-negative emissions from their land sectors.)

### 5.5. Heatmap

The Sankey diagram decomposed each country's emissions by sector and pollutant. The final chart drills down another level, allowing users to compare the sectoral shares of emissions for specific pollutants across countries. Each cell in the heatmap represents what share of a country's emissions of a specified gas come from that sector. The color scale runs from 0% to 100% for all gases.

First, let's define the basic chart settings.

In [20]:
HEATMAP_CMAP = cm.lipari       # change this to any colormap, e.g. cm.batlow, cm.navia, or matplotlib's plt.cm.viridis
HEATMAP_WIDTH, HEATMAP_HEIGHT = 900, 340
HEATMAP_SECTORS = ["1", "2", "M.AG", "4", "5"]      # top to bottom
HEATMAP_COUNTRIES = sorted(top10_name)   # the line-chart countries, A to Z

Next, the colors. I'll use the same `get_colors` utility function, but will trim off more at the tail ends of the colormap (especially on the lower end).

In [21]:
def colormap_to_hex(cmap=HEATMAP_CMAP, lo=.1, hi=.95, n=256):
    # A colormap object -> list of n hex colors, trimming low and high values
    return [to_hex(c) for c in cmap(np.linspace(lo, hi, n))]

Then the function to build the components of the heatmap. I'll need to need to normalize the emissions into percent shares.

In [22]:
# Prep the data
def heatmap_data(year, gas):
    # One row per country and sector with the sector's share of the country's total for this gas.
    d = ghg_sector_gas[(ghg_sector_gas["year"] == year) & (ghg_sector_gas["gas"] == gas)
                       & ghg_sector_gas["name"].isin(HEATMAP_COUNTRIES)]
    totals = d[d["sector"] == TOTAL_SECTOR].set_index("name")[VALUE_COL]
    parts = d[d["sector"].isin(HEATMAP_SECTORS)].copy()
    if parts.empty or totals.empty:
        raise ValueError(f"no data for gas {gas!r} in {year}")
    parts["share"] = (parts[VALUE_COL] / parts["name"].map(totals)
                      ).replace([np.inf, -np.inf], np.nan)            # a missing or zero total -> no share
    parts["sector_name"] = parts["sector"].map(sector_label)
    parts["share_text"] = parts["share"].map(lambda v: "---" if pd.isna(v) else f"{v:.1%}")   # text for the tooltip
    return parts[["name", "sector_name", "share", "share_text"]]

# Create the hooks for the chart
def make_heatmap_hook():
    tip = tooltip_box('<div style="font-weight: 600;">@name</div>'
                      '<div>@sector_name: <b>@share_text</b></div>')
    sector_order = [sector_label(c) for c in HEATMAP_SECTORS]

    def hook(plot, element):
        fig = plot.state
        keep_only_hover(fig)
        fig.outline_line_color = None
        fig.x_range.factors = HEATMAP_COUNTRIES                 # A to Z
        fig.y_range.factors = sector_order[::-1]                # Bokeh lists the bottom row first
        fig.axis.axis_line_color = None
        fig.axis.major_tick_line_color = None
        apply_font(fig.axis, "axis_labels", "major_label_text_")
        apply_font(fig.title, "title")
        for tool in fig.toolbar.tools:
            tool.tooltips = tip
        for bar in fig.right:                                    # the colour scale on the side
            if isinstance(bar, ColorBar):
                bar.ticker = FixedTicker(ticks=[0, 0.5, 1])      # labels at 0%, 50%, 100%
                bar.formatter = NumeralTickFormatter(format="0%")
                apply_font(bar, "legend", "major_label_text_")
                bar.major_tick_line_color = None
                bar.bar_line_color = None
                bar.width = 16
                bar.title = None
    return hook

# Create chart object
def make_heatmap(year, gas):
    # Heatmap of sector shares (rows) for every country (columns), for one year and one gas.
    return hv.HeatMap(heatmap_data(year, gas), kdims=["name", "sector_name"], vdims=["share", "share_text"]).opts(
        title=f"Share of {'total' if gas == 'total' else gas_label(gas)} emissions by sector, {year}",
        width=HEATMAP_WIDTH, height=HEATMAP_HEIGHT,
        cmap=colormap_to_hex(HEATMAP_CMAP), clim=(0, 1), clipping_colors={"NaN": "#e5e7eb"},        # colours always span 0% to 100%
        colorbar=True, xlabel="", ylabel="", tools=["hover"],
        line_color="white", line_width=2,                        # thin gaps between cells
        hooks=[make_heatmap_hook()],
    )

Here's a sample heatmap for the sectoral shares of methane emissions in 2025. Notice how fossil fuels are a bigger source of methane for countries like China (which mines a lot of coal) and Iran (which vents or leaks a lot of methane from its massive, aging network of wells and pipes). For more agricultural societies, such as Brazil and India, methane comes more from farming and animal husbandry.

For some of the gases, the vast majority of emissions come from just one sector (e.g., F-gases from industrial processes, or CO<sub>2</sub> from energy). The heatmap visualizations are less interesting/effective for plotting these sectors. I've trimmed the color map so that the colors aren't all white or all black for these cases.


In [23]:
heatmap_2025 = make_heatmap(2025, "ch4")
heatmap_2025

:HeatMap   [name,sector_name]   (share,share_text)

### 5.6. Interactivity

Now that the charts comprising the dashboard have been created, I'll introduce interactive elements such as sliders, dropdowns, and toggles.

#### 5.6.1. Dropdown and year-range slider for line plot and bar charts

The dropdown (total or per capita) and the "Years" range slider drive both charts. The line chart shows the selected years; the bar chart compares the earliest and latest selected year. A range needs at least two years, so if both handles sit on the same year the range is widened by one.

In [24]:
VARIABLE_OPTIONS = {"Total": "emissions", "Per capita": "emissions_percap"}   # label -> column

# Look shared by all the year sliders: dark grey handles, the selected year printed under each handle
SLIDER_COLOUR = "#595A5C"
SLIDER_WIDTH, RANGE_SLIDER_WIDTH = 150, 170       # pixels
def slider_look(label):
    # Shared look for the year sliders: dark grey handles, the selected year printed under each handle,
    # and a plain title (Panel would otherwise print "Years:" with a colon).
    css = f'''
    .bk-slider-title {{ font-size: 0; }}
    .bk-slider-title::before {{ content: "{label}"; font-size: 12px; }}
    .noUi-handle {{ background: {SLIDER_COLOUR}; border: none; box-shadow: none; }}
    .noUi-handle::before, .noUi-handle::after {{ display: none; }}
    .noUi-connect {{ background: {SLIDER_COLOUR}; }}
    .noUi-target {{ margin-bottom: 28px; }}
    .noUi-tooltip {{ display: block !important; top: 130%; bottom: auto; border: none; background: transparent;
                     padding: 0; color: {SLIDER_COLOUR}; font-size: 13px; font-weight: 600; }}
    '''
    return dict(name=label, bar_color=SLIDER_COLOUR, show_value=False, tooltips=True, format="0", stylesheets=[css])


variable_select = pn.widgets.Select(name="Show total or per-capita emissions", options=VARIABLE_OPTIONS,
                                    value="emissions", width=200)
year_range = pn.widgets.IntRangeSlider(start=FIRST, end=LAST, value=(FIRST, LAST), step=1,
                                        width=RANGE_SLIDER_WIDTH, **slider_look("Years"))
line_chart = pn.bind(make_line_chart, variable_select, year_range.param.value_throttled)
bar_chart = pn.bind(make_bar_chart, variable_select, year_range.param.value_throttled)

emissions_panel = pn.Column(pn.Row(variable_select, year_range), line_chart, bar_chart)


#### 5.6.2. Dropdown and slider for Sankey chart

The country list is set in `SANKEY_COUNTRIES`. The slider is released-to-update `value_throttled`), so dragging it does not redraw the chart on every step.

In [25]:
SANKEY_COUNTRIES = sorted(top10_name)
SANKEY_FIRST, SANKEY_LAST = int(ghg_sector_gas["year"].min()), int(ghg_sector_gas["year"].max())

sankey_country = pn.widgets.Select(name="Country", options=SANKEY_COUNTRIES, value="China", width=240)
sankey_year = pn.widgets.IntSlider(start=SANKEY_FIRST, end=SANKEY_LAST, value=SANKEY_LAST, step=1,
                                   width=SLIDER_WIDTH, **slider_look("Year"))
sankey_chart = pn.bind(make_sankey, sankey_country, sankey_year.param.value_throttled)

sankey_panel = pn.Column(pn.Row(sankey_country, sankey_year), sankey_chart)

#### 5.6.3. Dropdown and slider for heatmap

To select individual pollutants and years (which can be different from the years in the Sankey chart).

In [26]:
GAS_OPTIONS = {                       # label shown in the dropdown -> value in the `gas` column
    "All gases": "total",
    "Carbon dioxide (CO2)": "co2",
    "Methane (CH4)": "ch4",
    "Nitrous oxide (N2O)": "n2o",
    "F-gases": "fgas",
}
HEATMAP_FIRST, HEATMAP_LAST = int(ghg_sector_gas["year"].min()), int(ghg_sector_gas["year"].max())

heatmap_gas = pn.widgets.Select(name="Gas", options=GAS_OPTIONS, value="total", width=240)
heatmap_year = pn.widgets.IntSlider(start=HEATMAP_FIRST, end=HEATMAP_LAST, value=HEATMAP_LAST, step=1,
                                    width=SLIDER_WIDTH, **slider_look("Year"))
heatmap_chart = pn.bind(lambda gas, year: make_heatmap(year, gas), heatmap_gas, heatmap_year.param.value_throttled)

heatmap_panel = pn.Column(pn.Row(heatmap_gas, heatmap_year), heatmap_chart)

#### 5.6.4. Layout

Arrange the page text, style white cards on which the charts will be plotted, and determine where charts are placed. This last step assembles the dashboard for serving to [Render](https://render.com/), which I will use to host my web app

In [27]:
# ---- Page look ----------------------------------------------------------------------------------
PAGE_BACKGROUND = "#f5f2ed"       # very light warm grey behind the cards
CARD_BACKGROUND = "#ffffff"
CARD_WIDTH = 1300                 # maximum card width, in pixels
CARD_RADIUS = "18px"              # rounded corners
CARD_PADDING = "32px 36px"
CARD_GAP = "24px"                 # space between cards
SIDE_BY_SIDE_FROM = 1250          # windows wider than this (pixels) put the line and bar charts side by side
LINK_COLOR = "#2563eb"

FONT_FAMILY = "Helvetica, Arial, sans-serif"
PAGE_TEXT = {                     # one entry per kind of page text: size, weight (400 normal, 700 bold), colour
    "page_title":       {"size": "60px", "weight": "800", "color": highlight_colors[0]},
    "page_intro":       {"size": "18px", "weight": "400", "color": "#6b7280"},
    "section_title":    {"size": "32px", "weight": "800", "color": highlight_colors[0]},
    "section_subtitle": {"size": "17px", "weight": "400", "color": "#6b7280"},
    "chart_title":      {"size": "22px", "weight": "700", "color": highlight_colors[0]},
    "source":           {"size": "15px", "weight": "400", "color": "#6b7280"}
}

# ---- Page text ------------------------------------------------------------------------------------
PAGE_TITLE = "GHG Emissions Explorer"
PAGE_INTRO = (
    "Explore the greenhouse gas (GHG) emissions profiles for the top-10 largest countries using data from the "
    f"<a href='https://primap.org/primap-hist/' target='_blank' style='color:{LINK_COLOR}'>PRIMAP-hist</a> dataset by "
    "Johannes Gütschow and coauthors."
)

SECTION_1_TITLE = "Annual emissions for the top-10 largest countries, 1990-2025"
SECTION_1_SUBTITLE = (
    "These 10 countries and regions accounted for nearly three-quarters of global GHG emissions in 2025. China emits "
    "2.5x more climate pollution each year as the USA, although its per-capita emissions are lower and on par "
    "with Europe."
)

SECTION_2_TITLE = "Dissecting emissions by source sector and pollutant"
SECTION_2_SUBTITLE = (
    "Emissions of the primary GHGs, broken down by source sector, for the top-10 countries and regions. Most GHG "
    "emissions come from burning fossil fuels, although the source sectors and mix of climate pollutants emitted "
    "by these sectors vary considerably across these big countries and regions, reflecting their different energy "
    "systems and economies."
)
SANKEY_TITLE = "Emissions by source and gas for individual countries"
HEATMAP_TITLE = "Comparing sector shares of different gases across countries"

SOURCES_TITLE = "Sources"
SOURCES = [
    "Gütschow, J.; Pflüger, M. (2026): The PRIMAP-hist national historical emissions time series (1750-2025) v2.8. "
    "Zenodo. <a href='https://doi.org/10.5281/zenodo.22876287' target='_blank' "
    f"style='color:{LINK_COLOR}'>https://doi.org/10.5281/zenodo.22876287</a>",
    "Gütschow, J.; Jeffery, M. L.; Gieseke, R.; Gebel, R.; Stevens, D.; Krapp, M.; Rocha, M. (2016): The PRIMAP-hist "
    "national historical emissions time series. Earth System Science Data, 8, 571-603. "
    f"<a href='https://doi.org/10.5194/essd-8-571-2016' target='_blank' style='color:{LINK_COLOR}'>"
    "https://doi.org/10.5194/essd-8-571-2016</a>",
    "World Bank. Population, total (SP.POP.TOTL). World Development Indicators. "
    f"<a href='https://data.worldbank.org/indicator/SP.POP.TOTL' target='_blank' style='color:{LINK_COLOR}'>"
    "https://data.worldbank.org/indicator/SP.POP.TOTL</a>",
]
DATA_NOTE = "This dashboard was created by Chris Sall with help from Claude and is provided under an MIT license. Licenses of the underlying data sets may vary"

In [28]:
def styled_text(kind, html, tag="div", margin="0"):
    # A piece of page text in one of the PAGE_TEXT styles.
    t = PAGE_TEXT[kind]
    style = (f"font-family:{FONT_FAMILY}; font-size:{t['size']}; font-weight:{t['weight']}; color:{t['color']}; "
             f"line-height:1.35; margin:{margin};")
    return pn.pane.HTML(f"<{tag} style=\"{style}\">{html}</{tag}>", sizing_mode="stretch_width", margin=(0, 0))


def card(*items):
    # A white rounded card, at most CARD_WIDTH wide.
    return pn.Column(*items, sizing_mode="stretch_width", margin=(0, 0, 0, 0),
                     styles={"background": CARD_BACKGROUND, "border-radius": CARD_RADIUS,
                             "padding": CARD_PADDING, "box-sizing": "border-box",
                             "max-width": f"{CARD_WIDTH}px", "width": "100%",
                             "margin-left": "auto", "margin-right": "auto"})


def scrollable(chart):
    # Fixed-width charts get a scrollbar inside the card instead of spilling out of it on very small windows.
    return pn.Column(chart, sizing_mode="stretch_width", margin=0, styles={"overflow-x": "auto"})


def section_header(title, subtitle):
    return pn.Column(styled_text("section_title", title, "h2"),
                     styled_text("section_subtitle", subtitle, "p", margin="10px 0 0 0"),
                     sizing_mode="stretch_width", margin=(0, 0, 18, 0))


# Line and bar chart: side by side in wide windows, stacked in narrow ones
chart_pair = pn.Row(
    line_chart, bar_chart, sizing_mode="stretch_width",
    stylesheets=[f":host {{ display: flex; flex-direction: row; justify-content: center; gap: 20px; }} "
                 f"@media (max-width: {SIDE_BY_SIDE_FROM}px) {{ :host {{ flex-direction: column; align-items: center; }} }}"])

In [29]:
header = pn.Column(
    styled_text("page_title", PAGE_TITLE, "h1", margin="0 0 14px 0"),
    styled_text("page_intro", PAGE_INTRO, "p"),
    sizing_mode="stretch_width", margin=(10, 0, 0, 0),
    styles={"padding": "0 8px", "box-sizing": "border-box", "max-width": f"{CARD_WIDTH}px", "width": "100%",
            "margin-left": "auto", "margin-right": "auto"})

section_1 = card(
    section_header(SECTION_1_TITLE, SECTION_1_SUBTITLE),
    pn.Row(variable_select, year_range),
    chart_pair,
)

section_2 = card(
    section_header(SECTION_2_TITLE, SECTION_2_SUBTITLE),
    styled_text("chart_title", SANKEY_TITLE, "h3", margin="0 0 10px 0"),
    pn.Row(sankey_country, sankey_year),
    scrollable(sankey_chart),
    styled_text("chart_title", HEATMAP_TITLE, "h3", margin="28px 0 10px 0"),
    pn.Row(heatmap_gas, heatmap_year),
    scrollable(heatmap_chart),
)

sources_items = "".join(f"<li style='margin-bottom:8px'>{ref}</li>" for ref in SOURCES)
sources_note = f"<p style='margin:14px 0 0 0'><i>{DATA_NOTE}</i></p>" if DATA_NOTE else ""
section_3 = card(
    styled_text("section_title", SOURCES_TITLE, "h2", margin="0 0 14px 0"),
    styled_text("source", f"<ul style='margin:0; padding-left:20px'>{sources_items}</ul>{sources_note}"),
)

if pn.state.served:      # when served as a web app, also colour the area around the cards
    pn.config.raw_css.append(f"body {{ background: {PAGE_BACKGROUND}; margin: 0; }}")

dashboard = pn.Column(
    header, section_1, section_2, section_3,
    sizing_mode="stretch_width", margin=0,
    styles={"background": PAGE_BACKGROUND, "padding": f"24px 16px 48px 16px", "gap": CARD_GAP,
            "box-sizing": "border-box"})
dashboard.servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'984152d0-2e9f-4ece-8184-4524737e968f': {'version…

## 6. AI Log

I built the dashboard with a lot of help from Claude. A full transcript of the Claude chat can be found [here](https://drive.google.com/file/d/1WRc08c17E6_yfa5nNw9SGC-o9FKDYwU0/view?usp=sharing). A summary of what I asked Claude to do is below.

**Entry 1:** Summarize the menu of suggested libraries
- **Tool**: Claude
- **What I asked for**: The instructions for this assignment provide a laundry list of Python libraries, some for plotting, some for the widgets/controls to build the dashboard, and some for hosting web apps. I'm familiar with some of these libraries but was unsure which are the most straightforward for integrating into Jupyter. I asked Claude to introduce the libraries, provide URLs for them (including for plotting library galleries), and explain which work well with Jupyter.
- **What came back**: Claude identified Panel/HoloViews, Dash, Gradio, Solara, and Voilà as working in notebooks. It said that Streamlit, Shiny, marimo, and Observable aren't usually run inside Jupyter. It noted that Render offers free hosting for web apps.
- **What was wrong or could be better**: Nothing 
- **How I noticed**: I checked the Panel and HoloView sites and looked through their galleries. Many of the examples are provided as Jupyter notebooks and seemed to be similar to the use case for this assignment. Panel/HoloView also offer flexibility with plotting libraries, so I could create some charts with hvPlot, some with seaborn, and some with Plotly. 
- **What I did**: I decided to go with Panel/Holoview mainly because I wanted to build the dashboard within this Jupyter notebook.

**Entry 2:** Make line chart
- **Tool**: Claude
- **What I asked for**: I asked Claude to build each of the dashboard's four visualizations, chart by chart. I started by asking it to make a line chart of countries' total emisions for 1990-2025 using HoloViews/hvPlot with interactive tooltips and highlights on hover, with minimalist chart styling.
- **What came back**: Claude wrote functions for all the chart hooks, styling, and plot elements.
- **What was wrong or could be better**: 
    - `make_line_chart` originally drew line plots for all countries that weren't in the `HIGHLIGHT` list, so I ended up with 200+ plots. 
    - Horizontal grid lines added visual clutter and bled over the edge of the chart area.
    - Changed tooltip labels to one decimal point.
    - Removed the rectangular border around the chart.
    - The toolbar was hidden (per my prompt to use minimalist styling), but scrolling with a mouse over the chart area still zoomed the chart and center.
    - The colors were Claude's standard chart colors. 
- **How I noticed**: I ran the functions and inspected the resulting line plots.
- **What I did**:
    - I added a mask in `make_chart` to only include the top-10 countries (those in the `top10_name` list).
    - I asked Claude to remove the zoom upon scrolling.
    - I removed y-axis grid lines.
    - I also adjusted the fonts and sizes.
    - I adjusted the xlim property to avoid cutting off country name label for Saudi Arabia.
    - I used Fabio Crameri's scientific color maps to build a unified color scheme, adapting a function called `get_colors` that I had created for MADS SIADS 522. I asked for Claude's help in explaining how a the colormap objects from cmcamari could be divided into intervals. (It turns out they're just Matplotlib colormap objects.)

**Entry 3:** Make bar chart
- **Tool**: Claude
- **What I asked for**: Make a bar chart using HoloViews/hvPlot with minimalist chart styling.
- **What came back**: Claude reused the `apply_minimal_style` and `make_tooltip` functions that it created for the line charts. It then defined new `make_bar_hook` and `make_bar_chart` functions (the ones seen above).
- **What was wrong or could be better**: As with the line chart, the bar chart function returned bar charts for all 200+ countries. The bars were originally just grey and black
- **How I noticed**: I ran the the functions and inspected the resulting line plots.
- **What I did**:
    - I added a mask for the top-10 countries, as with the line chart.
    - I used my `get_colors` function to adjust the colormap and scheme to make the coloring consistent but not exactly the same as the line plot.
    
**Entry 4:** Slice a color map 
- **Tool**: Claude
- **What I asked for**: How to slice up a Matplotlib colormap into even intervals.
- **What came back**: A function using `np.linspace` to slice up a color map into fractions. 
- **What was wrong or could be better**: The function worked fine. 
- **How I noticed**: I ran it and inspected the outputs.
- **What I did**: I added `lo` and `hi` parameters to the function and then incorporated it into a function I had already scripted. This cleared up for me whether the `cmcrameri` colormaps were discrete sequences or continuous colormaps.

**Entry 5:** Create Sankey diagram / alluvial chart
- **Tool**: Claude
- **What I asked for**: Use Panel and HoloViews to create an alluvial chart.
- **What came back**: HoloViews doesn't have a `Alluvial` element, so Claude built the chart like a Sankey diagram.
- **What was wrong or could be better**: 
    - Overly complicated method and functions for defining node colors
    - Overly complicated method for checking if node totals properly summed (i.e., that the summed total of emissions across the different gases equaled total emissions as reported at the source)
- **How I noticed**:
    - I wondered why Claude had added such complicated checks for comparing totals by gas and sector, so I went back and looked at the data.  I added the emissions totals for the sectors and gases for China and discovered a discrepancy, which it turns out was due to rounding in the source data. PRIMAP-hist offers both rounded and unrounded versions of the data. I switched to the version without rounding, and the totals all matched up.
- **What I did**:
    - I drastically simplified the method for defining colors and introduced the option for using two different color maps for the sector and gas dimensions of the diagram.
    - I updated the source data to the version without rounding. 
    
**Entry 6:** Create heatmap
- **Tool**: Claude
- **What I asked for**: Use Panel and HoloViews to create a heatmap.
- **What came back**: Claude wrote functions for all the chart hooks, styling, and plot elements.
- **What was wrong or could be better**: 
    - The chart functions filled `NaN` values with zeros, which was misleading because some countries just don't estimate emissions for some specific sectors. 
    - The hooks and styling for the charts used the full range of the colormaps for the share of emissions from specific sectors (0-100%). The distributions were lumpy, and for some sectors, the shares were all close to 90-100% or 0-10%. This made it hard to tell the differences between values at the tails. 
- **How I noticed**: I inspected the charts for eacho f the gases
- **What I did**:
    - I trimmed the colormap to make the variation in the tails easier to see.

**Entry 7:** Add interactive elements
- **Tool**: Claude
- **What I asked for**: I asked Claude to create the dropdown menus, toggles, and sliders for all the charts. 
- **What came back**: Functions for each of these interactive elements, created as IPython widgets and bound to the chart objects using Panel.
- **What was wrong or could be better**: The appearance of some of the elements looked off. 
- **How I noticed**: I looked at the rendered charts in my notebook
- **What I did**: I adjusted the sizes, contents, and colors of sliders and dropdowns.

**Entry 8:** Overall layout
- **Tool**: Claude
- **What I asked for**: I asked Claude to piece together the overall layout of the dashboard 
- **What came back**: The last three cells. The layout (titles, subtitles, section intros) are a combination of HTML text, inserted into Panel building blocks (`Panes`', `Columns`, and `Rows`). Panel then combines the layout, panes, and widgets and calls the `.servable()` function to tell the server which UI components to show.  
- **What was wrong or could be better**: I asked Claude to use dictionaries to make it easy to adjust fonts, sizes, and colors for the different text elements in the panes and columns. I didn't like some of the style choices, so I tinkered with them and made adjustments. 
- **How I noticed**: I looked at the rendered dashboard in my notebook
- **What I did**: I adjusted the sizes, colors, and styling of text elements on the page.

**Entry 9:** Hosting on Render
- **Tool**: Claude
- **What I asked for**: I asked Claude for step-by-step instructions on how to upload and host my dashboard on Render.
- **What came back**: Claude provided instructions and created a `render.yaml` file, which allowed me to start a new service on Render using my GitHub repo and a blueprint.   
- **What was wrong or could be better**: I had some issues at first with large input data sets being downloaded by Render and then eating up most of the available memory (512MB) for my app. This was because I forgot to fix a reference in my `.gitignore` file to point to the specific CSV in the data folder that I wanted to exclude (rather than the entire data folder).  
- **How I noticed**: I tried to deploy the app a few times and failed. I asked Claude to look at my GitHub repo and tell me why the app was failing to deploy.
- **What I did**: I fixed the `gitignore` file and the incorrectly defined `Path` variable in my notebook pointing to my data folder.


## 7. Instructions for deploying the app

I deployed my dashboard using Render's free service for hosting web apps. You can see the final app [here](https://ghg-explorer-svpx.onrender.com/).

You can create an app on Render using *my* GitHub repo for this dashboard and *your* Render account by following these steps:
- Sign in at render.com. If you signed up with GitHub, the repo connection is already made.
- Click **+ New** and select **Web Service.**
- Under **Source Code,** pick **Public GitHub Repository** and provide the URL for my repo: https://github.com/sallcj/ghg-emissions-explorer, then **Connect->**.
- Fill in the settings:
    - **Name**: this sets your URL (https://<name>.onrender.com), so choose it deliberately. Remember it for the start command
    - **Language**: Python 3.
    - **Branch**: main
    - **Root Directory**: leave blank
    - **Build Command**: pip install -r requirements.txt
    - **Start Command**: `panel serve dashboard.ipynb --address 0.0.0.0 --port $PORT --allow-websocket-origin=<name>.onrender.com`
    - **Instance Type**: Free.
    - Add an environment variable `PYTHON_VERSION` and set it to `3.13.5`, or any 3.13 version Render lists.
- **Deploy Web Service**. The first build takes a few minutes. Watch the Logs tab until you see the server start.
